# CPython 运行机制

学习目标：用小型观察区分 Python 语言规则与 CPython 的字节码、导入缓存和对象回收机制，并据此理解 GIL 的作用与版本边界。

前置知识：名称绑定与对象引用、函数和类、模块导入、异常处理、with、线程与锁。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例在结束时关闭线程池，并恢复垃圾回收开关。

## 1 语言规则与实现细节

Python 是编程语言，CPython 是它的一个实现，也是 python.org 分发的标准实现。“Python 对象如何表现”和“CPython 内部怎样实现”需要分别判断。

名称绑定与对象引用已经用于解释容器共享；本章关注这些规则背后的 CPython 机制及其适用范围。

| 名称 | 中文名称／含义 | 本章的判断边界 |
| --- | --- | --- |
| Python | 编程语言 | 例如 is 比较对象身份，两个新建列表是不同对象 |
| CPython | Python 的标准实现 | 采用引用计数，并补充循环垃圾回收 |
| bytecode | 字节码 | CPython 执行代码的内部表示，不保证跨版本稳定 |

可移植代码应依赖语言规定的行为。不要把某些整数或字符串恰好共享对象的观察，推广为所有实现都必须如此。

In [1]:
first_notes = ["导入"]
shared_notes = first_notes
copied_notes = list(first_notes)
shared_notes.append("回收")

print(first_notes)  # ['导入', '回收']：两个名称引用同一个列表。
print(first_notes is shared_notes)  # True：共享身份。
print(first_notes is copied_notes)  # False：复制产生另一个列表。
print(copied_notes)  # ['导入']：没有跟随原列表追加。

['导入', '回收']
True
False
['导入']


## 2 编译得到代码对象，执行才产生结果

compile 把源码编译成代码对象；filename 参数给出诊断用名称，不要求创建同名文件。mode="exec" 表示一组语句，mode="eval" 表示一个表达式。

下面只使用本章写死的可信源码。exec 执行代码对象，传入一个字典作为执行命名空间，便于观察赋值。这个字典用于组织名称，不是安全沙箱，不能把任意外部输入交给 exec。

In [2]:
source = "total = minutes + 5"
compiled = compile(source, "<lesson-total>", "exec")
namespace = {"minutes": 40}

print("total" in namespace)  # False：编译尚未执行赋值。
exec(compiled, namespace)
print(namespace["total"])  # 45：执行时才读取 minutes 并绑定 total。
namespace["minutes"] = 10
exec(compiled, namespace)
print(namespace["total"])  # 15：同一代码对象可以使用新的输入执行。

False
45
15


## 3 用 dis 观察字节码

dis.dis 显示反汇编，dis.get\_instructions 逐条提供指令信息。下面函数的参数 minutes 表示学习分钟数；操作码（opcode）表示执行动作，操作数进一步指定变量或运算。

| Python 3.12 操作码 | 中文名称／含义 |
| --- | --- |
| LOAD\_FAST | 读取已初始化的局部变量 |
| LOAD\_CONST | 读取代码对象中的常量 |
| BINARY\_OP | 执行由操作数指定的二元或原地运算 |
| RETURN\_VALUE | 返回栈顶值 |

CPython 3.12 的指令、偏移和显示格式属于实现细节。自适应专门化（specialization）会按运行情况调整部分指令；adaptive=True 可查看专门化形式，show\_caches=True 可显示默认隐藏的内联缓存。本例使用默认视图，不把某个指令序列或专门化时机写成跨版本保证。

In [3]:
import dis


def add_break(minutes: int) -> int:
    """在学习分钟数后增加五分钟休息。"""
    return minutes + 5


print(add_break(40))  # 45：先核对函数行为，再读取实现层面的指令。
dis.dis(add_break)  # 显示指令偏移与操作名；本例加载 minutes、加 5、返回，行号随排版变化。
operations = [item.opname for item in dis.get_instructions(add_break)]
print(operations)
# 在本章 CPython 3.12 默认视图中观察加载、加法和返回。
# 指令数不能直接当作耗时；不要为兼容未来版本断言完整操作码列表。

45
  4           0 RESUME                   0

  6           2 LOAD_FAST                0 (minutes)
              4 LOAD_CONST               1 (5)
              6 BINARY_OP                0 (+)
             10 RETURN_VALUE
['RESUME', 'LOAD_FAST', 'LOAD_CONST', 'BINARY_OP', 'RETURN_VALUE']


## 4 导入先查模块缓存

导入先按完整模块名查询 sys.modules：已有模块对象时直接使用；缺少该键才继续查找；键存在但值为 None 时抛出 ModuleNotFoundError。删除本地导入名称不会删除模块缓存，也不等于卸载模块。

缓存未命中后，导入机制沿 sys.meta\_path 调用查找器（finder）。查找器提供模块规格，加载器（loader）负责加载并执行模块代码；默认机制也能处理内置模块，并非只在磁盘上搜索 .py 文件。

| API 或属性 | 中文名称／含义 |
| --- | --- |
| sys.modules | 完整模块名到已导入模块对象的缓存 |
| sys.path | 路径查找器使用的搜索位置列表 |
| sys.path\_importer\_cache | 搜索位置到路径入口查找器的缓存，保存的不是模块对象 |
| importlib.invalidate\_caches | 通知查找器清理内部查找缓存 |

运行期间新建模块文件时，可能需要 invalidate\_caches 使查找器发现它。这个操作不会清除 sys.modules，也不会重新执行已缓存模块；不要把两种缓存混为一谈。

In [4]:
import importlib
import sys

first_math = importlib.import_module("math")
second_math = importlib.import_module("math")
print(first_math is second_math)  # True：同名导入命中模块缓存。
del second_math
print(sys.modules["math"] is first_math)  # True：删除的只是本地名称。

importlib.invalidate_caches()
third_math = importlib.import_module("math")
print(third_math is first_math)  # True：查找缓存失效没有移除已导入模块。
print(third_math.sqrt(81))  # 9.0：缓存对象仍可使用。

True
True
True
9.0


### 4.1 缓存中的 None 表示阻止导入

下面只临时占用一个专用模块名，并在 finally 中删除该条目。用这个反例区分“字典里没有这个名称”和“这个名称对应 None”；不删除或替换其他已加载模块。

In [5]:
blocked_name = "lesson28_blocked_module"
assert blocked_name not in sys.modules
sys.modules[blocked_name] = None
try:
    try:
        importlib.import_module(blocked_name)
    except ModuleNotFoundError as error:
        assert error.name == blocked_name
        print(type(error).__name__)  # ModuleNotFoundError：缓存明确阻止导入。
    else:
        raise AssertionError("None 缓存条目没有阻止导入")
finally:
    del sys.modules[blocked_name]

print(blocked_name not in sys.modules)  # True：专用条目已清理。

ModuleNotFoundError
True


## 5 引用计数只是实现层面的诊断

CPython 用引用计数（reference counting）跟踪对象引用。给对象增加别名或把它放进容器会增加引用；del 删除的是名称或容器项，不能据此断言对象一定销毁。

sys.getrefcount 自身传参通常带来一个临时引用，调试器、交互环境等也可能额外持有对象。Python 3.12 还引入不朽对象（immortal objects）：它们的很大计数不代表实际引用数量。因此不能把返回值当作业务逻辑依据，也不能据此给所有对象编写固定计数断言。

下面在函数局部观察一个新建普通实例，只比较增加和移除别名前后的变化，不规定绝对计数。

In [6]:
class StudyNote:
    """保存一条可被弱引用的学习笔记。"""

    def __init__(self, title: str) -> None:
        self.title = title


def observe_alias() -> tuple[bool, bool]:
    """观察普通实例增加和移除局部别名时的引用计数变化。"""
    note = StudyNote("引用")
    before = sys.getrefcount(note)
    alias = note
    with_alias = sys.getrefcount(note)
    del alias
    after = sys.getrefcount(note)
    return with_alias > before, after == before


print(observe_alias())  # 本例为 (True, True)，不承诺某个绝对引用数。

(True, True)


## 6 弱引用不延长对象寿命

weakref.ref 创建弱引用（weak reference），它允许观察对象，却不会仅因这个引用而阻止回收。调用弱引用时，对象仍存活就返回对象，否则返回 None；并非所有类型都支持弱引用，普通用户自定义类实例支持本例用法。

如果把弱引用的调用结果保存在变量中，这个新变量又会成为强引用。下面先保留一个明确的强引用，再去掉它并主动收集；不借此承诺任意对象的自动终结时机。

In [7]:
import gc
import weakref

note = StudyNote("弱引用")
observer = weakref.ref(note)
keeper = note

del note
print(observer() is keeper)  # True：keeper 仍持有强引用。
del keeper
gc.collect()
print(observer() is None)  # True：本例没有其他强引用，也没有终结器。
# 不把 observer() 的结果另存为变量，避免观察动作延长目标寿命。

True


True

## 7 循环引用需要额外检测

两个对象相互引用，或对象引用自身，即使外部名称都已删除，循环内部仍有引用。CPython 的 gc 模块补充检测这种不可达循环；关闭自动循环收集不会关闭引用计数。

下面短暂关闭自动收集，保证能先观察到自引用对象尚存，再用 gc.collect 主动收集。返回的回收数量可能包含其他垃圾，不能拿它断言“恰好回收一个对象”。finally 恢复原开关，不改变阈值、调试标志或回收回调。

Python 不保证对象变得不可达后立即终结；外部资源仍应使用 with 或显式 close，不能把文件关闭责任交给垃圾回收时机。

In [8]:
class LinkedNote:
    """用 next 属性表示笔记之间的引用关系。"""

    def __init__(self) -> None:
        self.next: LinkedNote | None = None


def observe_cycle() -> tuple[bool, bool]:
    """观察自引用对象在手动循环收集前后的存活状态。"""
    was_enabled = gc.isenabled()
    gc.disable()
    try:
        note = LinkedNote()
        note.next = note
        # 弱引用只用于观察；删除外部强引用后，自引用环仍然存在。
        observer = weakref.ref(note)
        del note
        alive_before = observer() is not None
        gc.collect()
        gone_after = observer() is None
        return alive_before, gone_after
    finally:
        if was_enabled:
            gc.enable()
        else:
            gc.disable()


original_gc_state = gc.isenabled()
print(observe_cycle())  # (True, True)：循环内部引用需要循环收集处理。
print(gc.isenabled() == original_gc_state)  # True：恢复进入时的开关。

(True, True)
True


## 8 GIL 不保证业务操作线程安全

全局解释器锁（global interpreter lock，GIL）使本章 CPython 3.12 同一解释器中的线程在同一时刻只能由一个执行 Python 字节码。它不把“读取值、计算、写回”这类复合业务操作自动变成不可分割的事务；共享状态仍需要锁等同步机制。

| 工作类型 | 中文名称／含义 | CPython 3.12 的考虑 |
| --- | --- | --- |
| CPU-bound Python code | 主要执行 Python 计算的任务 | 增加线程不等于让字节码在多个核上并行，可考虑进程 |
| I/O-bound work | 主要等待输入输出的任务 | 阻塞 I/O 会释放 GIL，线程可在等待期间交错推进 |
| C extension | C 扩展执行的工作 | 部分压缩或哈希等扩展在计算时释放 GIL，应核对具体实现 |

下面只演示用锁保护累计次数，不运行靠运气触发的无锁反例，也不以此测量线程加速。线程池的 with 会等待任务结束并释放资源，读取 Future.result 也使任务异常向调用方传播。

In [9]:
import concurrent.futures
import threading


class StudyCounter:
    """用同一把锁保护学习次数的累计。"""

    def __init__(self) -> None:
        self.value = 0
        self._lock = threading.Lock()

    def add_many(self, count: int) -> None:
        """累计指定次数，每次读取、加一和写回都在锁内。"""
        for _ in range(count):
            with self._lock:
                self.value += 1


counter = StudyCounter()
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    jobs = [executor.submit(counter.add_many, 2000) for _ in range(2)]
    for job in jobs:
        job.result(timeout=10)

print(counter.value)  # 4000：两个任务的累计全部保留。
assert counter.value == 4000

4000

## 9 自由线程的版本边界

自由线程（free-threading）是允许关闭 GIL 的 CPython 构建方式，不能通过修改本章普通 Python 3.12 的一个变量获得。

| CPython 版本 | 自由线程状态 | 使用条件 |
| --- | --- | --- |
| 3.12 | 本章使用带 GIL 的常规构建 | 本章没有运行自由线程实验 |
| 3.13 | 实验性支持，默认不启用 | 需要单独的自由线程可执行文件或相应构建 |
| 3.14 | 正式支持、不再属于实验功能，仍是可选构建 | 不代表默认构建已经改为无 GIL |

即使使用自由线程构建，运行时仍可能启用 GIL；不支持自由线程的 C 扩展也可能触发重新启用。构建支持、实际 GIL 状态和依赖兼容是不同条件。自由线程也不取消共享业务状态的同步责任。

## 本章小结

（1）语言规定对象的可观察行为；CPython 的字节码、引用计数和 GIL 则有实现与版本边界。

（2）compile 产生代码对象，exec 才执行；dis 用于观察实现，指令数量不能替代计时。

（3）sys.modules 保存模块对象，查找器缓存帮助发现模块；查找缓存失效不等于重新导入。

（4）弱引用不保活，循环引用由 gc 补充检测；不要依赖固定引用计数或即时终结来管理资源。

（5）GIL 不代替业务锁；3.13 与 3.14 的自由线程状态不同，且仍须确认构建和扩展兼容。

自查：能否说明一次实际观察属于语言规则、3.12 实现细节，还是特定运行状态？

## 练习

（1）先预测下面三个布尔值，再运行核对。解释删除本地名称后，为什么仍可以通过别名和缓存访问模块；核对标准是预测与输出一致，并能区分本地绑定与模块缓存。

In [10]:
exercise_math = importlib.import_module("math")
exercise_alias = exercise_math
print(exercise_math is sys.modules["math"])
del exercise_math
print(exercise_alias is importlib.import_module("math"))
print("exercise_math" in globals())
# 先记录预测，运行后逐项核对名称绑定与缓存条目的区别。

True
True
False


（2）把赋值语句 total = left + right 编译一次，在两个独立命名空间中分别给出整数输入 10、20 和 3、4。检查执行前都没有 total，执行后分别为 30 和 7，两个命名空间互不影响。

用 dis 显示这个代码对象，指出读取输入、加法和保存结果的位置。检查程序行为，不把完整操作码列表写成未来版本必须通过的断言。

In [11]:
exercise_source = "total = left + right"
# 在此编译一次，创建两个命名空间，分别执行并检查结果。
# 仅使用上面给定的可信源码；不读取外部输入交给 exec。

（3）分别构造一个没有循环的 LinkedNote 和一个自引用的 LinkedNote，用弱引用观察删除外部名称后的状态。短暂关闭自动循环收集，在删除名称前确认两者存活，再手动收集并检查两者最终均已失效。

在本章 CPython 3.12 普通实例条件下，无循环对象不需要等待循环收集；自引用对象在主动收集前仍可观察到。用 finally 恢复原有 gc 开关，并在原开关为开、关两种情况下分别核对恢复行为。不要保存弱引用返回的目标，也不要断言 gc.collect 的回收数量。

In [12]:
# 在此创建两种引用结构并用弱引用检查，不添加 __del__ 终结器。
# 进入和退出实验时比较 gc.isenabled()，确保未改变调用者的开关。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12；自由线程概览另列 3.13、3.14） | 3.12：[CPython 定义](https://docs.python.org/3.12/glossary.html#term-CPython)、[对象身份、引用计数与终结边界](https://docs.python.org/3.12/reference/datamodel.html#objects-values-and-types)、[字节码](https://docs.python.org/3.12/glossary.html#term-bytecode)、[compile](https://docs.python.org/3.12/library/functions.html#compile)、[exec 与命名空间](https://docs.python.org/3.12/library/functions.html#exec)；[dis 与版本变化](https://docs.python.org/3.12/library/dis.html)、[反汇编与专门化视图](https://docs.python.org/3.12/library/dis.html#dis.dis)、[指令迭代](https://docs.python.org/3.12/library/dis.html#dis.get_instructions)、[LOAD\_FAST](https://docs.python.org/3.12/library/dis.html#opcode-LOAD_FAST)、[LOAD\_CONST](https://docs.python.org/3.12/library/dis.html#opcode-LOAD_CONST)、[BINARY\_OP](https://docs.python.org/3.12/library/dis.html#opcode-BINARY_OP)、[RETURN\_VALUE](https://docs.python.org/3.12/library/dis.html#opcode-RETURN_VALUE)；[模块缓存与 None](https://docs.python.org/3.12/reference/import.html#the-module-cache)、[查找器与加载器](https://docs.python.org/3.12/reference/import.html#finders-and-loaders)、[路径查找器及缓存](https://docs.python.org/3.12/reference/import.html#the-path-based-finder)、[查找缓存失效](https://docs.python.org/3.12/library/importlib.html#importlib.invalidate_caches)；[getrefcount、临时引用与不朽对象](https://docs.python.org/3.12/library/sys.html#sys.getrefcount)、[弱引用与支持类型](https://docs.python.org/3.12/library/weakref.html)、[weakref.ref](https://docs.python.org/3.12/library/weakref.html#weakref.ref)、[gc 的循环收集与引用计数关系](https://docs.python.org/3.12/library/gc.html)、[收集开关](https://docs.python.org/3.12/library/gc.html#gc.disable)、[开关查询](https://docs.python.org/3.12/library/gc.html#gc.isenabled)、[手动收集](https://docs.python.org/3.12/library/gc.html#gc.collect)；[GIL 与 I/O、扩展](https://docs.python.org/3.12/glossary.html#term-global-interpreter-lock)、[复合操作与线程安全](https://docs.python.org/3.12/faq/library.html#what-kinds-of-global-value-mutation-are-thread-safe)、[CPU 任务与进程](https://docs.python.org/3.12/faq/library.html#can-t-we-get-rid-of-the-global-interpreter-lock)、[锁](https://docs.python.org/3.12/library/threading.html#lock-objects)、[线程池关闭](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Executor.shutdown)、[任务结果与异常](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Future.result)。版本概览：[3.13 实验性自由线程](https://docs.python.org/3.13/whatsnew/3.13.html#free-threaded-cpython)、[3.14 正式支持但仍可选](https://docs.python.org/3.14/whatsnew/3.14.html#free-threaded-python-is-officially-supported)、[3.14 运行时 GIL 与扩展条件](https://docs.python.org/3.14/howto/free-threading-python.html#the-global-interpreter-lock-in-free-threaded-python)、[3.14 同步责任](https://docs.python.org/3.14/howto/free-threading-python.html#thread-safety)。 |